<a href="https://colab.research.google.com/github/Kaynart/Projeto-Pipeline-PROUNI-BD26.1-gp08/blob/main/notebooks/ETL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [69]:
pip install pandas

In [70]:
import pandas as pd

# 1 Extração dos Dados

In [71]:
# Dicionário com os caminhos dos arquivos do ProUni
arquivos = {
    '2017': 'data/pda-prouni-2017.csv',
    '2018': 'data/pda-prouni-2018.csv',
    '2019': 'data/pda-prouni-2019.csv'
}

lista_dataframes = [] # Armazenará os DataFrames temporariamente

for ano, arquivo in arquivos.items():
    try:
        # Leitura configurada estritamente para o padrão dos dados abertos do MEC
        df = pd.read_csv(arquivo, sep=';', encoding='latin-1')

        # Cria a coluna de controle utilizando a chave do dicionário
        df['ano_origem'] = ano

        lista_dataframes.append(df)

    except Exception as e:
        print(f"Erro ao ler os dados do ano {ano}: {e}")

df = pd.concat(lista_dataframes, ignore_index=True) #Junrando os DFs
print(f"Total de linhas no df unificado: {len(df)}")

Total de linhas no df unificado: 718700


# 2 Transformação dos dados

In [72]:
# Analisando o padrão do dataset
display(df.head(5))
display(df.info())

,ï»¿ANO_CONCESSAO_BOLSA,CODIGO_EMEC_IES_BOLSA,NOME_IES_BOLSA,TIPO_BOLSA,MODALIDADE_ENSINO_BOLSA,NOME_CURSO_BOLSA,NOME_TURNO_CURSO_BOLSA,CPF_BENEFICIARIO_BOLSA,SEXO_BENEFICIARIO_BOLSA,RACA_BENEFICIARIO_BOLSA,DT_NASCIMENTO_BENEFICIARIO,BENEFICIARIO_DEFICIENTE_FISICO,REGIAO_BENEFICIARIO_BOLSA,SIGLA_UF_BENEFICIARIO_BOLSA,MUNICIPIO_BENEFICIARIO_BOLSA,ano_origem
0,2017.0,10.0,PONTIFÃCIA UNIVERSIDADE CATÃLICA DO PARANÃ,BOLSA PARCIAL 50%,Presencial,CiÃªncias ContÃ¡beis,Noturno,***72056999**,F,Branca,29/04/1992,N,Sul,PR,SAO JOSE DOS PINHAIS,2017
1,2017.0,10.0,PONTIFÃCIA UNIVERSIDADE CATÃLICA DO PARANÃ,BOLSA PARCIAL 50%,Presencial,CiÃªncias ContÃ¡beis,Noturno,***74440980**,M,Branca,24/01/1999,N,Sul,PR,TOLEDO,2017
2,2017.0,10.0,PONTIFÃCIA UNIVERSIDADE CATÃLICA DO PARANÃ,BOLSA PARCIAL 50%,Presencial,CiÃªncias ContÃ¡beis,Noturno,***78689980**,M,Branca,19/05/1994,N,Sul,PR,CURITIBA,2017
3,2017.0,10.0,PONTIFÃCIA UNIVERSIDADE CATÃLICA DO PARANÃ,BOLSA PARCIAL 50%,Presencial,CiÃªncias ContÃ¡beis,Noturno,***90299907**,M,Branca,31/01/1997,N,Sul,PR,TOLEDO,2017
4,2017.0,10.0,PONTIFÃCIA UNIVERSIDADE CATÃLICA DO PARANÃ,BOLSA PARCIAL 50%,Presencial,CiÃªncias ContÃ¡beis,Noturno,***90445902**,M,Branca,24/11/1994,N,Sul,PR,CURITIBA,2017


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 718700 entries, 0 to 718699
Data columns (total 16 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   ï»¿ANO_CONCESSAO_BOLSA          703223 non-null  float64
 1   CODIGO_EMEC_IES_BOLSA           703223 non-null  float64
 2   NOME_IES_BOLSA                  703223 non-null  object 
 3   TIPO_BOLSA                      703223 non-null  object 
 4   MODALIDADE_ENSINO_BOLSA         703223 non-null  object 
 5   NOME_CURSO_BOLSA                703223 non-null  object 
 6   NOME_TURNO_CURSO_BOLSA          703223 non-null  object 
 7   CPF_BENEFICIARIO_BOLSA          703223 non-null  object 
 8   SEXO_BENEFICIARIO_BOLSA         703223 non-null  object 
 9   RACA_BENEFICIARIO_BOLSA         703223 non-null  object 
 10  DT_NASCIMENTO_BENEFICIARIO      703223 non-null  object 
 11  BENEFICIARIO_DEFICIENTE_FISICO  703223 non-null  object 
 12  REGIAO_BENEFICIA

None

In [73]:
df = df.drop(columns='ano_origem')

### 2.1 Valores ausentes

In [74]:
print("Quantidade de instâncias ausentes por coluna:")
print(df.isnull().sum())

Quantidade de instâncias ausentes por coluna:
ï»¿ANO_CONCESSAO_BOLSA            15477
CODIGO_EMEC_IES_BOLSA             15477
NOME_IES_BOLSA                    15477
TIPO_BOLSA                        15477
MODALIDADE_ENSINO_BOLSA           15477
NOME_CURSO_BOLSA                  15477
NOME_TURNO_CURSO_BOLSA            15477
CPF_BENEFICIARIO_BOLSA            15477
SEXO_BENEFICIARIO_BOLSA           15477
RACA_BENEFICIARIO_BOLSA           15477
DT_NASCIMENTO_BENEFICIARIO        15477
BENEFICIARIO_DEFICIENTE_FISICO    15477
REGIAO_BENEFICIARIO_BOLSA         15477
SIGLA_UF_BENEFICIARIO_BOLSA       15477
MUNICIPIO_BENEFICIARIO_BOLSA      15477
dtype: int64


Todas as colunas, exceto pelo ano_origem, possuem a mesma quantidade de valores ausentes, ou seja, há chance de que esses valores ausentes ocorram nas mesmas instâncias. Caso essa hipótese esteja correta, basta remover essas instâncias de pouca informação.

In [77]:
# Análise dos valores nulos
mascara_nulos = df.isnull().all(axis=1)

linhas_completamente_nulas = df[mascara_nulos]

print("Linhas que são totalmente nulas:")
print(linhas_completamente_nulas)

Linhas que são totalmente nulas:
        ï»¿ANO_CONCESSAO_BOLSA  CODIGO_EMEC_IES_BOLSA NOME_IES_BOLSA  \
703223                     NaN                    NaN            NaN   
703224                     NaN                    NaN            NaN   
703225                     NaN                    NaN            NaN   
703226                     NaN                    NaN            NaN   
703227                     NaN                    NaN            NaN   
...                        ...                    ...            ...   
718695                     NaN                    NaN            NaN   
718696                     NaN                    NaN            NaN   
718697                     NaN                    NaN            NaN   
718698                     NaN                    NaN            NaN   
718699                     NaN                    NaN            NaN   

       TIPO_BOLSA MODALIDADE_ENSINO_BOLSA NOME_CURSO_BOLSA  \
703223        NaN                     Na

Para essa análise, a coluna ano_origem foi desconsiderada por ser imputada manualmente ao unificar os datasets. Com isso, a hipótese se mostrou correta e todos os 15477 valores nulos tratam-se das mesmas linhas, as quais devem ser removidas.

In [79]:
# Descrição das colunas para que a remoção das instâncias nulas seja controlada
total_nulas = df[df.columns].isnull().all(axis=1).sum()
print(f"Quantidade de linhas que serão removidas: {total_nulas}")

df.dropna(subset=df.columns, how='all',inplace=True)

Quantidade de linhas que serão removidas: 0


In [80]:
# Confirmação da remoção das colunas
print(df.isnull().sum())

ï»¿ANO_CONCESSAO_BOLSA            0
CODIGO_EMEC_IES_BOLSA             0
NOME_IES_BOLSA                    0
TIPO_BOLSA                        0
MODALIDADE_ENSINO_BOLSA           0
NOME_CURSO_BOLSA                  0
NOME_TURNO_CURSO_BOLSA            0
CPF_BENEFICIARIO_BOLSA            0
SEXO_BENEFICIARIO_BOLSA           0
RACA_BENEFICIARIO_BOLSA           0
DT_NASCIMENTO_BENEFICIARIO        0
BENEFICIARIO_DEFICIENTE_FISICO    0
REGIAO_BENEFICIARIO_BOLSA         0
SIGLA_UF_BENEFICIARIO_BOLSA       0
MUNICIPIO_BENEFICIARIO_BOLSA      0
dtype: int64


### 2.2 Valores únicos

Nessa parte, a análise é sobre quais colunas têm atributos de mesmo valor, mas com escritas diferentes (exemplo: OUTROS e OUTRAS). Além disso, também será analisada a localidade dos dados, se cumpre o requisito do projeto ter dados apenas de Pernambuco.

In [81]:
# Identifica valores únicos
df.nunique(dropna=False)

,0
ï»¿ANO_CONCESSAO_BOLSA,3
CODIGO_EMEC_IES_BOLSA,1485
NOME_IES_BOLSA,1469
TIPO_BOLSA,2
MODALIDADE_ENSINO_BOLSA,2
NOME_CURSO_BOLSA,2266
NOME_TURNO_CURSO_BOLSA,5
CPF_BENEFICIARIO_BOLSA,549249
SEXO_BENEFICIARIO_BOLSA,2
RACA_BENEFICIARIO_BOLSA,6


In [82]:
# Análise de valores únicos
print("Valores únicos da coluna 'Ano de concessão da bolsa': \n", df['ï»¿ANO_CONCESSAO_BOLSA'].unique())
print()

print("Valores únicos da coluna 'Código EMEC': \n", df['CODIGO_EMEC_IES_BOLSA'].value_counts().head(5))
print()

print("Valores únicos da coluna 'Nome IES': \n", df['NOME_IES_BOLSA'].value_counts().head(5))
print()

print("Valores únicos da coluna 'Tipo de bolsa': \n", df['TIPO_BOLSA'].unique())
print()

print("Valores únicos da coluna 'Modalidade de Ensino Bolsa': \n", df['MODALIDADE_ENSINO_BOLSA'].unique())
print()

print("Valores únicos da coluna 'Nome do curso': \n", df['NOME_CURSO_BOLSA'].value_counts().head(5))
print()

print("Valores únicos da coluna 'Turno do curso': \n", df['NOME_TURNO_CURSO_BOLSA'].unique())

Valores únicos da coluna 'Ano de concessão da bolsa': 
 [2017. 2018. 2019.]

Valores únicos da coluna 'Código EMEC': 
 CODIGO_EMEC_IES_BOLSA
322.0     68539
1491.0    29943
298.0     25296
316.0     14518
671.0     13937
Name: count, dtype: int64

Valores únicos da coluna 'Nome IES': 
 NOME_IES_BOLSA
UNIVERSIDADE PAULISTA                  68539
CENTRO UNIVERSITÃRIO INTERNACIONAL    29943
UNIVERSIDADE PITÃGORAS UNOPAR         25296
UNIVERSIDADE NOVE DE JULHO             14518
UNIVERSIDADE ANHANGUERA                13937
Name: count, dtype: int64

Valores únicos da coluna 'Tipo de bolsa': 
 ['BOLSA PARCIAL 50%' 'BOLSA INTEGRAL']

Valores únicos da coluna 'Modalidade de Ensino Bolsa': 
 ['Presencial' 'EAD']

Valores únicos da coluna 'Nome do curso': 
 NOME_CURSO_BOLSA
AdministraÃ§Ã£o         53285
Direito                 49916
Pedagogia               48004
CiÃªncias ContÃ¡beis    32462
Enfermagem              27549
Name: count, dtype: int64

Valores únicos da coluna 'Turno do curso': 
 

In [83]:
# Divisão em dois blocos para melhor visualização
print("Valores únicos da coluna 'CPF do beneficiário': \n", df['CPF_BENEFICIARIO_BOLSA'].value_counts().head(5))
print()

print("Valores unicos da coluna 'Sexo do beneficiário': \n", df['SEXO_BENEFICIARIO_BOLSA'].unique())
print()

print("Valores únicos da coluna 'Raça/Cor do beneficiário': \n", df['RACA_BENEFICIARIO_BOLSA'].unique())
print()

print("Valores únicos da coluna 'Data de nascimento do beneficiário': \n", df['DT_NASCIMENTO_BENEFICIARIO'].unique())
print()

print("Valores únicos da coluna 'Beneficiário é deficiente físico': \n", df['BENEFICIARIO_DEFICIENTE_FISICO'].unique())
print()

print("Valores unicos da coluna 'Região do beneficiário': \n", df['REGIAO_BENEFICIARIO_BOLSA'].unique())
print()

print("Valores únicos da coluna 'Sigla UF do beneficiário': \n", df['SIGLA_UF_BENEFICIARIO_BOLSA'].value_counts().head(5))
print()

print("Valores unicos da coluna 'Município do beneficiário': \n", df['MUNICIPIO_BENEFICIARIO_BOLSA'].value_counts().head(5))

Valores únicos da coluna 'CPF do beneficiário': 
 CPF_BENEFICIARIO_BOLSA
***12884025**    12
***21978486**    12
***55193851**    12
***90782040**    12
***91044692**    11
Name: count, dtype: int64

Valores unicos da coluna 'Sexo do beneficiário': 
 ['F' 'M']

Valores únicos da coluna 'Raça/Cor do beneficiário': 
 ['Branca' 'Parda' 'Preta' 'Amarela' 'IndÃ\xadgena' 'NÃ£o Informada']

Valores únicos da coluna 'Data de nascimento do beneficiário': 
 ['29/04/1992' '24/01/1999' '19/05/1994' ... '11/04/1969' '28/12/1954'
 '24/02/1963']

Valores únicos da coluna 'Beneficiário é deficiente físico': 
 ['N' 'S']

Valores unicos da coluna 'Região do beneficiário': 
 ['Sul' 'Sudeste' 'Norte' 'Nordeste' 'Centro-Oeste']

Valores únicos da coluna 'Sigla UF do beneficiário': 
 SIGLA_UF_BENEFICIARIO_BOLSA
SP    184313
MG     81097
RS     49654
PR     48207
BA     42989
Name: count, dtype: int64

Valores unicos da coluna 'Município do beneficiário': 
 MUNICIPIO_BENEFICIARIO_BOLSA
SAO PAULO         5316

A análise dos valores repetidos de CPF não indica que necessariamente há repetição de participantes. Como o número do CPF é parcialmente ocultado, é possível que a sequência do meio seja igual, mas com diferença no início ou no final. Não foi identificada nenhum desvio de escrita entre os datasets que exigisse alteração.

### 2.3 Correção de tipagem de atributos

Com base nas visualizações da etapa anterior, é possível classificar corretamente o tipo de cada atributo e fazer a conversão necessária.

In [84]:
# São as duas únicas colunas cujo tipo é float, estão sendo convertidas para inteiros
df['ï»¿ANO_CONCESSAO_BOLSA'] = pd.to_numeric(df['ï»¿ANO_CONCESSAO_BOLSA'], errors='coerce').astype('Int64')
df['CODIGO_EMEC_IES_BOLSA'] = pd.to_numeric(df['CODIGO_EMEC_IES_BOLSA'], errors='coerce').astype('Int64')

In [85]:
# Conversão para string
df['NOME_IES_BOLSA'] = df['NOME_IES_BOLSA'].astype(str)
df['TIPO_BOLSA'] = df['TIPO_BOLSA'].astype(str)
df['MODALIDADE_ENSINO_BOLSA'] = df['MODALIDADE_ENSINO_BOLSA'].astype(str)
df['NOME_CURSO_BOLSA'] = df['NOME_CURSO_BOLSA'].astype(str)
df['NOME_TURNO_CURSO_BOLSA'] = df['NOME_TURNO_CURSO_BOLSA'].astype(str)
df['RACA_BENEFICIARIO_BOLSA'] = df['RACA_BENEFICIARIO_BOLSA'].astype(str)
df['REGIAO_BENEFICIARIO_BOLSA'] = df['REGIAO_BENEFICIARIO_BOLSA'].astype(str)
df['MUNICIPIO_BENEFICIARIO_BOLSA'] = df['MUNICIPIO_BENEFICIARIO_BOLSA'].astype(str)

In [86]:
# Deve ter seu tamanho limitado a 2 ao converter
df['SIGLA_UF_BENEFICIARIO_BOLSA'] = df['SIGLA_UF_BENEFICIARIO_BOLSA'].astype(str)

# Deve ter seu tamanho limitado a 11 ao converter
# Como há ocultação, NÃO pode ser convertido para inteiro
df['CPF_BENEFICIARIO_BOLSA'] = df['CPF_BENEFICIARIO_BOLSA'].astype(str)

In [87]:
# Conversão para string, mas devem ser tratados como CHAR ao converter
df['SEXO_BENEFICIARIO_BOLSA'] = df['SEXO_BENEFICIARIO_BOLSA'].astype(str)
df['BENEFICIARIO_DEFICIENTE_FISICO'] = df['BENEFICIARIO_DEFICIENTE_FISICO'].astype(str)

In [88]:
# Aqui a data de nascimento é convertida para datetime
# É importante ressaltar que o formato utilizado é DD/MM/YYYY
df['DT_NASCIMENTO_BENEFICIARIO'] = pd.to_datetime(df['DT_NASCIMENTO_BENEFICIARIO'], format='%d/%m/%Y', errors='coerce')

# Caso seja necessário dividir a data em dia, mês e ano,
# essa parte deve ser alterada e dessa coluna devem surgir outras 3

### 2.4 Correção de escrita

Os dados foram carregados de forma que caracteres como acentos e cedilhas não foram identificados na maneira certa, resultado em palavras com erro de escrita. Tais erros serão corrigidos aqui.

In [90]:
# Esse vai demorar mais para corrigir, já que são 1469 nomes únicos que podem estar escritos de forma errada
# df['NOME_IES_BOLSA'] = df['NOME_IES_BOLSA'].replace('escrita correta')

# Já esse são 2266 nomes
# df['NOME_CURSO_BOLSA'] = df['NOME_CURSO_BOLSA'].replace('escrita correta')

In [94]:
df['NOME_TURNO_CURSO_BOLSA'] = df['NOME_TURNO_CURSO_BOLSA'].replace('Curso a distÃ¢ncia', 'Curso a distância')

In [93]:
df['RACA_BENEFICIARIO_BOLSA'] = df['RACA_BENEFICIARIO_BOLSA'].replace('IndÃ\xadgena', 'Indígena')
df['RACA_BENEFICIARIO_BOLSA'] = df['RACA_BENEFICIARIO_BOLSA'].replace('NÃ£o Informada', 'Não Informada')

Por mais que haja nomes de municípios com acentos, eles não foram escritos no dataset, ou seja, o que normalmente seria "São Paulo" e teria erro de digitação, já foi escrito como "SAO PAULO".

### 2.5 Possíveis alterações

Este tópico é mais um lembrete de coisas que talvez sejam alteradas:
1. A escrita de algumas palavras está diferente por questão de formatação no csv. Quando há acentos ou outros caracteres especiais, aparecem outros símbolos.
2. Data nascimento talvez seja segregada em dia, mês e ano.